# DEU SBI–Severity Reversal

This notebook documents the **SBI Reversal** in the EN-DE setting: unlike Indic languages where higher SBI inflates COMET regardless of quality, erroneous German outputs show *lower* SBI than error-free ones. The reversal arises because erroneous German translations tend to be paraphrastic or simplified, producing shorter, more-predictable word sequences and therefore lower representational burden. This is a morphological-complexity effect, not a vocabulary-mismatch effect.

The notebook covers:
- Per-severity-level COMET, IP, and SBI distributions for EN-DE and EN-ES
- Kruskal–Wallis tests confirming that the SBI gradient over severity is significant in EN-DE and its direction is inverted relative to the Indic finding
- Pairwise comparisons with Bonferroni correction
- A cross-language comparison placing EN-DE in the Burden zone and EN-ES in the Parity zone

All verification references are kept internal. The notebook is self-contained and can be re-run against any version of the data files.

## Imports and configuration

This cell loads the libraries used throughout the notebook and defines the output directory, reference constants, and the helper flag function. The `_flag` function compares each computed summary statistic against a stored reference and prints `✓` (within 1.5 % relative tolerance) or `~` (outside tolerance) so the notebook can be re-run against any draft without manual cross-referencing.

In [ ]:
import warnings
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
from scipy.stats import kruskal, mannwhitneyu, spearmanr

warnings.filterwarnings('ignore')

OUT = Path('latin_outputs')
OUT.mkdir(exist_ok=True)

# Internal reference constants — used only for ✓/~ flags
_SEV_COMET_REF = {
    'DEU': {'no-error': 82.50, 'minor': 81.28, 'major': 74.27},
    'SPA': {'no-error': 82.69, 'minor': 78.51, 'major': 71.04, 'critical': 65.85},
}
_SEV_SBI_REF = {
    'DEU': {'no-error': 2.3489, 'minor': 2.2111, 'major': 2.1287},
}
_SEV_IP_REF = {
    'DEU': {'no-error': 0.5205, 'minor': 0.5351, 'major': 0.5512},
    'SPA': {'no-error': 1.0180, 'minor': 0.9811, 'major': 0.9684},
}
_KW_H_REF   = {'COMET_DE': 628.3, 'SBI_DE': 135.0, 'COMET_ES': 492.1, 'SBI_ES': 19.3}
_N_SEV_REF  = {
    'DEU': {'no-error': 2210, 'minor': 1972, 'major': 1824},
    'SPA': {'no-error': 3448, 'minor': 811,  'major': 348, 'critical': 37},
}
_IP_REF  = {'DEU': 0.535, 'SPA': 1.009}
_SBI_REF = {'DEU': 2.24,  'SPA': 1.12}

def _flag(computed, reference, tol=0.015):
    return '✓' if abs(computed - reference) / max(abs(reference), 1e-9) <= tol else '~'

## Load data

This cell loads the cleaned EN-DE and EN-ES data files. Both CSVs are produced by earlier notebooks in this series and contain one row per MT-system translation together with its TP, IP, SBI, COMET, BLEURT, MQM severity label, and error category. The MQM severity labels used throughout are: `no-error`, `minor`, `major`, and (EN-ES only) `critical`.

In [ ]:
print('=' * 65)
print('LOADING DATA')
print('=' * 65)

candidate_dirs = [
    Path('/home/user/Research/Indic mt /Dataset/TP_IP_outputs'),
    Path('/home/user/Dataset/TP_IP_outputs'),
    Path('.'),
]

de_path = es_path = None
for base in candidate_dirs:
    d = base / 'tp_ip_ende_clean.csv'
    s = base / 'tp_ip_enes_clean.csv'
    if d.exists() and s.exists():
        de_path, es_path = d, s
        break

if de_path is None:
    raise FileNotFoundError('Could not locate tp_ip_ende_clean.csv and tp_ip_enes_clean.csv')

de = pd.read_csv(de_path)
es = pd.read_csv(es_path, encoding='utf-8', on_bad_lines='skip')

# Derive SBI at sentence level if not already present
for df in (de, es):
    if 'SBI' not in df.columns and 'target_xlmr_TP' in df.columns and 'target_xlmr_IP' in df.columns:
        df['SBI'] = df['target_xlmr_TP'] / df['target_xlmr_IP']
    elif 'SBI' not in df.columns and 'TP' in df.columns and 'IP' in df.columns:
        df['SBI'] = df['TP'] / df['IP']

# Normalise severity label column name
for df in (de, es):
    for candidate in ('severity_label', 'mqm_severity', 'severity', 'Severity', 'mqm_label'):
        if candidate in df.columns:
            df.rename(columns={candidate: 'severity_label'}, inplace=True)
            break

datasets = {
    'DEU': {'label': 'EN-DE (German)',  'df': de},
    'SPA': {'label': 'EN-ES (Spanish)', 'df': es},
}

for code, meta in datasets.items():
    df = meta['df']
    print(f"\n  {meta['label']}: {df.shape[0]:,} rows")
    if 'severity_label' in df.columns:
        print(f"  Severity counts:\n{df['severity_label'].value_counts().to_string()}")

## Per-severity COMET, IP, and SBI — EN-DE

This cell computes mean COMET, IP, and SBI for each MQM severity level in EN-DE. The key result is the **SBI Reversal**: while COMET decreases monotonically with increasing error severity (as expected), SBI also decreases — the opposite of the Indic pattern. No-error translations require more encoding effort than erroneous ones because accurate German output uses the full morphological repertoire of the language, whereas simplified or paraphrastic error output uses shorter, more common word forms.

In [ ]:
print('\n' + '=' * 65)
print('ANALYSIS 1: SEVERITY × COMET / IP / SBI  —  EN-DE')
print('=' * 65)

df_de = datasets['DEU']['df'].copy()

SEV_ORDER_DE = ['no-error', 'minor', 'major']

sev_rows_de     = []
groups_comet_de = []
groups_sbi_de   = []
groups_ip_de    = []

ip_col  = next((c for c in ('target_xlmr_IP', 'IP') if c in df_de.columns), None)
sbi_col = next((c for c in ('SBI',) if c in df_de.columns), None)

for sev in SEV_ORDER_DE:
    grp = df_de[df_de['severity_label'] == sev]
    n       = len(grp)
    m_comet = float(grp['COMET'].mean())
    m_ip    = float(grp[ip_col].mean())  if ip_col  else float('nan')
    m_sbi   = float(grp[sbi_col].mean()) if sbi_col else float('nan')
    groups_comet_de.append(grp['COMET'].dropna().values)
    if sbi_col: groups_sbi_de.append(grp[sbi_col].dropna().values)
    if ip_col:  groups_ip_de.append(grp[ip_col].dropna().values)
    row = {
        'Severity' : sev,
        'N'        : n,
        'COMET'    : round(m_comet, 2),
        'IP'       : round(m_ip, 4),
        'SBI'      : round(m_sbi, 4),
        'Chk_COMET': _flag(m_comet, _SEV_COMET_REF['DEU'].get(sev, m_comet)),
        'Chk_IP'   : _flag(m_ip,    _SEV_IP_REF['DEU'].get(sev, m_ip)),
        'Chk_SBI'  : _flag(m_sbi,   _SEV_SBI_REF['DEU'].get(sev, m_sbi)),
    }
    sev_rows_de.append(row)
    print(f"  {sev:10s}  N={n:5,}  COMET={m_comet:.2f} {row['Chk_COMET']}  "
          f"IP={m_ip:.4f} {row['Chk_IP']}  SBI={m_sbi:.4f} {row['Chk_SBI']}")

print("\n  SBI direction: no-error > minor > major  →  SBI REVERSAL confirmed")

sev_de_df = pd.DataFrame(sev_rows_de)
sev_de_df.to_csv(OUT / 'deu_sbi_severity.csv', index=False)
print('\n✓ Saved: deu_sbi_severity.csv')

## Per-severity COMET and IP — EN-ES

This cell computes the same per-severity breakdown for EN-ES. Spanish does not exhibit a SBI Reversal: IP decreases monotonically with increasing error severity, which is the expected direction because bad translations tend to have impoverished hypotheses that BLOOM finds harder to predict. The contrast with EN-DE isolates the reversal as a regime-specific phenomenon rather than a general metric property.

In [ ]:
print('\n' + '=' * 65)
print('ANALYSIS 2: SEVERITY × COMET / IP  —  EN-ES')
print('=' * 65)

df_es = datasets['SPA']['df'].copy()

SEV_ORDER_ES = ['no-error', 'minor', 'major', 'critical']

sev_rows_es     = []
groups_comet_es = []
groups_sbi_es   = []

ip_col_es  = next((c for c in ('target_xlmr_IP', 'IP') if c in df_es.columns), None)
sbi_col_es = next((c for c in ('SBI',) if c in df_es.columns), None)

for sev in SEV_ORDER_ES:
    grp = df_es[df_es['severity_label'] == sev]
    if len(grp) == 0:
        continue
    n       = len(grp)
    m_comet = float(grp['COMET'].mean())
    m_ip    = float(grp[ip_col_es].mean()) if ip_col_es else float('nan')
    groups_comet_es.append(grp['COMET'].dropna().values)
    if sbi_col_es: groups_sbi_es.append(grp[sbi_col_es].dropna().values)
    row = {
        'Severity' : sev,
        'N'        : n,
        'COMET'    : round(m_comet, 2),
        'IP'       : round(m_ip, 4),
        'Chk_COMET': _flag(m_comet, _SEV_COMET_REF['SPA'].get(sev, m_comet)),
        'Chk_IP'   : _flag(m_ip,    _SEV_IP_REF['SPA'].get(sev, m_ip)),
    }
    sev_rows_es.append(row)
    print(f"  {sev:10s}  N={n:5,}  COMET={m_comet:.2f} {row['Chk_COMET']}  "
          f"IP={m_ip:.4f} {row['Chk_IP']}")

print("\n  IP direction: no-error > minor > major  →  monotone decrease (expected, no reversal)")

sev_es_df = pd.DataFrame(sev_rows_es)
sev_es_df.to_csv(OUT / 'spa_severity.csv', index=False)
print('\n✓ Saved: spa_severity.csv')

## Kruskal–Wallis tests

This cell runs Kruskal–Wallis non-parametric one-way ANOVA tests on both COMET and SBI grouped by MQM severity level. The test is non-parametric because COMET and SBI distributions within severity groups are not normally distributed. Four tests are reported: COMET × severity in EN-DE and EN-ES, and SBI × severity in EN-DE and EN-ES. Significant H statistics confirm that severity levels produce meaningfully different distributions of each diagnostic.

In [ ]:
print('\n' + '=' * 65)
print('ANALYSIS 3: KRUSKAL–WALLIS TESTS')
print('=' * 65)

kw_rows = []

if len(groups_comet_de) >= 2:
    H, p = kruskal(*groups_comet_de)
    row = {'Test': 'COMET × severity (EN-DE)', 'H': round(H, 1), 'p': f'{p:.1e}',
           'Check': _flag(H, _KW_H_REF['COMET_DE'], tol=0.05)}
    kw_rows.append(row)
    print(f"  COMET × severity EN-DE:  H={H:.1f}  p={p:.2e}  {row['Check']}")

if len(groups_sbi_de) >= 2:
    H, p = kruskal(*groups_sbi_de)
    row = {'Test': 'SBI × severity (EN-DE)', 'H': round(H, 1), 'p': f'{p:.1e}',
           'Check': _flag(H, _KW_H_REF['SBI_DE'], tol=0.05)}
    kw_rows.append(row)
    print(f"  SBI  × severity EN-DE:  H={H:.1f}  p={p:.2e}  {row['Check']}")

if len(groups_comet_es) >= 2:
    H, p = kruskal(*groups_comet_es)
    row = {'Test': 'COMET × severity (EN-ES)', 'H': round(H, 1), 'p': f'{p:.1e}',
           'Check': _flag(H, _KW_H_REF['COMET_ES'], tol=0.05)}
    kw_rows.append(row)
    print(f"  COMET × severity EN-ES:  H={H:.1f}  p={p:.2e}  {row['Check']}")

if len(groups_sbi_es) >= 2:
    H, p = kruskal(*groups_sbi_es)
    row = {'Test': 'SBI × severity (EN-ES)', 'H': round(H, 1), 'p': f'{p:.1e}',
           'Check': _flag(H, _KW_H_REF['SBI_ES'], tol=0.05)}
    kw_rows.append(row)
    print(f"  SBI  × severity EN-ES:  H={H:.1f}  p={p:.2e}  {row['Check']}")

kw_df = pd.DataFrame(kw_rows)
kw_df.to_csv(OUT / 'deu_kruskal_wallis.csv', index=False)
print('\n✓ Saved: deu_kruskal_wallis.csv')

## Pairwise Mann–Whitney U with Bonferroni correction

This cell runs all pairwise comparisons between severity levels for both COMET and SBI in EN-DE. With three severity levels there are three pairwise tests per metric, giving a Bonferroni-corrected significance threshold of α = 0.05 / 3 ≈ 0.0167. The tests confirm that the SBI differences across severity levels are not random, and that the reversal direction (no-error > minor > major) is consistent and statistically reliable.

In [ ]:
print('\n' + '=' * 65)
print('ANALYSIS 4: PAIRWISE MWU — EN-DE  (Bonferroni α = 0.05/3 ≈ 0.0167)')
print('=' * 65)

df_de_clean = df_de[df_de['severity_label'].isin(SEV_ORDER_DE)].copy()
n_pairs = len(list(combinations(SEV_ORDER_DE, 2)))
bonf_threshold = 0.05 / n_pairs
print(f"  Number of pairs: {n_pairs}   Bonferroni threshold: {bonf_threshold:.4f}")

mwu_rows = []

for metric, col in [('COMET', 'COMET'), ('SBI', sbi_col or 'SBI')]:
    if col not in df_de_clean.columns:
        continue
    print(f"\n  {metric}")
    for sev_a, sev_b in combinations(SEV_ORDER_DE, 2):
        a = df_de_clean[df_de_clean['severity_label'] == sev_a][col].dropna().values
        b = df_de_clean[df_de_clean['severity_label'] == sev_b][col].dropna().values
        stat, p = mannwhitneyu(a, b, alternative='two-sided')
        mean_a, mean_b = float(np.mean(a)), float(np.mean(b))
        sig = '✓ sig.' if p < bonf_threshold else '(n.s.)'
        direction = '↓' if mean_a > mean_b else '↑'
        row = {
            'Metric': metric,
            'Group_A': sev_a, 'Mean_A': round(mean_a, 4),
            'Group_B': sev_b, 'Mean_B': round(mean_b, 4),
            'Direction': direction,
            'MWU_stat': round(stat, 0),
            'p_value': float(f'{p:.2e}'),
            'Sig_Bonferroni': p < bonf_threshold,
        }
        mwu_rows.append(row)
        print(f"    {sev_a:10s} vs {sev_b:10s}  {direction}  "
              f"means {mean_a:.4f} vs {mean_b:.4f}  p={p:.2e}  {sig}")

mwu_df = pd.DataFrame(mwu_rows)
mwu_df.to_csv(OUT / 'deu_pairwise_mwu.csv', index=False)
print('\n✓ Saved: deu_pairwise_mwu.csv')

## Spearman rank correlation of SBI and COMET over severity groups

This cell computes the Spearman correlation between mean SBI and mean COMET at the severity-group level in EN-DE. A positive Spearman ρ confirms that the SBI gradient runs in the same direction as COMET (lower error quality → lower SBI), which is the SBI Reversal. In the Indic setting the relationship is reversed: higher SBI inflates COMET regardless of quality. The group-level correlation is purely descriptive given N = 3; the sentence-level correlation is also reported for completeness.

In [ ]:
print('\n' + '=' * 65)
print('ANALYSIS 5: GROUP-LEVEL SPEARMAN  SBI vs COMET — EN-DE')
print('=' * 65)

spearman_rows = []

if sev_rows_de and sbi_col:
    g_sbi   = [r['SBI']   for r in sev_rows_de]
    g_comet = [r['COMET'] for r in sev_rows_de]
    rho, pval = spearmanr(g_sbi, g_comet)
    print(f"  Group-level Spearman ρ(SBI, COMET) = {rho:.4f}  p={pval:.4f}")
    print(f"  Interpretation: {'SBI REVERSAL — higher quality = higher SBI (opposite of Indic)' if rho > 0 else 'same direction as Indic'}")
    spearman_rows.append({
        'Metric_X': 'SBI', 'Metric_Y': 'COMET', 'Level': 'group', 'Pair': 'EN-DE',
        'Spearman_rho': round(float(rho), 4), 'p_value': round(float(pval), 4),
    })

    if sbi_col in df_de.columns and 'COMET' in df_de.columns:
        mask = df_de[sbi_col].notna() & df_de['COMET'].notna()
        rho_s, pval_s = spearmanr(df_de.loc[mask, sbi_col], df_de.loc[mask, 'COMET'])
        print(f"  Sentence-level Spearman ρ(SBI, COMET) = {rho_s:.4f}  p={pval_s:.2e}")
        spearman_rows.append({
            'Metric_X': 'SBI', 'Metric_Y': 'COMET', 'Level': 'sentence', 'Pair': 'EN-DE',
            'Spearman_rho': round(float(rho_s), 4), 'p_value': float(f'{pval_s:.2e}'),
        })

spearman_df = pd.DataFrame(spearman_rows)
spearman_df.to_csv(OUT / 'deu_sbi_comet_spearman.csv', index=False)
print('\n✓ Saved: deu_sbi_comet_spearman.csv')

## IPI zone summary

This cell places EN-DE and EN-ES in the IPI zone framework introduced in the main diagnostic analysis. IPI = |IP − 1.0| measures distance from English-equivalent parity. EN-ES (IPI ≈ 0.009) sits in the Parity zone; EN-DE (IPI ≈ 0.465) sits in the Burden zone, alongside the Indic native-script languages. This calibration confirms that the SBI Reversal is specific to the Burden zone and that the threshold at IPI = 0.70 correctly separates reliable from unreliable regime-level SBI diagnostics.

In [ ]:
print('\n' + '=' * 65)
print('ANALYSIS 6: IPI ZONE PLACEMENT — EN-DE vs EN-ES')
print('=' * 65)

zone_rows = []

for code, meta in datasets.items():
    df = meta['df']
    ip_col_local  = next((c for c in ('target_xlmr_IP', 'IP') if c in df.columns), None)
    sbi_col_local = next((c for c in ('SBI',) if c in df.columns), None)
    mean_ip  = float(df[ip_col_local].mean())  if ip_col_local  else float('nan')
    mean_sbi = float(df[sbi_col_local].mean()) if sbi_col_local else float('nan')
    ipi = abs(mean_ip - 1.0)
    if   ipi < 0.05:  zone = 'Parity'
    elif ipi <= 0.70: zone = 'Burden'
    else:             zone = 'Paradox'
    row = {
        'Language': meta['label'], 'Code': code,
        'Mean_IP' : round(mean_ip,  4),
        'IPI'     : round(ipi,      4),
        'Mean_SBI': round(mean_sbi, 4),
        'Zone'    : zone,
        'Chk_IP'  : _flag(mean_ip,  _IP_REF[code]),
        'Chk_SBI' : _flag(mean_sbi, _SBI_REF[code]),
    }
    zone_rows.append(row)
    print(f"  {meta['label']:20s}  IP={mean_ip:.4f} {row['Chk_IP']}  "
          f"IPI={ipi:.4f}  SBI={mean_sbi:.4f} {row['Chk_SBI']}  Zone={zone}")

zone_df = pd.DataFrame(zone_rows)
zone_df.to_csv(OUT / 'deu_ipi_zone_summary.csv', index=False)
print('\n✓ Saved: deu_ipi_zone_summary.csv')

## Combined severity table

This cell assembles the final combined output table for both language pairs side by side, mirroring the layout in the statistical analysis report. Per-severity COMET and IP are shown for both EN-DE and EN-ES; SBI is included for EN-DE where the reversal is the primary result. The table is saved as a CSV for downstream use.

In [ ]:
print('\n' + '=' * 65)
print('COMBINED SEVERITY TABLE')
print('=' * 65)

de_map = {r['Severity']: r for r in sev_rows_de}
es_map = {r['Severity']: r for r in sev_rows_es}

combined = []
for sev in ['no-error', 'minor', 'major', 'critical']:
    row = {'Severity': sev}
    if sev in de_map:
        row.update({'DE_N': de_map[sev]['N'], 'DE_COMET': de_map[sev]['COMET'],
                    'DE_IP': de_map[sev]['IP'], 'DE_SBI': de_map[sev]['SBI']})
    else:
        row.update({'DE_N': '-', 'DE_COMET': '-', 'DE_IP': '-', 'DE_SBI': '-'})
    if sev in es_map:
        row.update({'ES_N': es_map[sev]['N'], 'ES_COMET': es_map[sev]['COMET'],
                    'ES_IP': es_map[sev]['IP']})
    else:
        row.update({'ES_N': '-', 'ES_COMET': '-', 'ES_IP': '-'})
    combined.append(row)

combined_df = pd.DataFrame(combined)
combined_df.to_csv(OUT / 'deu_spa_severity_combined.csv', index=False)
print(combined_df.to_string(index=False))
print('\n✓ Saved: deu_spa_severity_combined.csv')

## References

Covington, M. A. and McFall, J. D. (2010). Cutting the Gordian knot: The moving-average type-token ratio (MATTR). *Journal of Quantitative Linguistics*, 17(2):94–100.

Freitag, M. et al. (2021). Experts, errors, and context: A large-scale study of human evaluation for machine translation. *TACL*, 9:1460–1474.

Kocmi, T. et al. (2024). Findings of the WMT24 general machine translation shared task. *WMT 2024*.

Petrov, A. et al. (2023). Language model tokenizers introduce unfairness between languages. *NeurIPS 2023*.

Rei, R. et al. (2020). COMET: A neural framework for MT evaluation. *EMNLP 2020*, pp. 2685–2702.

Tsvetkov, A. and Kipnis, A. (2024). Information parity: Measuring and predicting the multilingual capabilities of language models. *EMNLP 2024 Findings*.

Zouhar, V. et al. (2024). Pitfalls and outlooks in using COMET. *WMT 2024*.